In [12]:
import ee
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Authenticate and initialize GEE
# ==========================================
ee.Authenticate()

True

In [3]:
ee.Initialize(project="sundarbans-climate-analysis")

In [4]:
# 2. Define study area
# Approximate Sundarbans coastal region
# ==========================================
roi = ee.Geometry.Rectangle(
    [88.5, 20.5, 90.5, 22.0]
)

In [5]:
# 3. Load HYCOM data
# ==========================================
collection = (
    ee.ImageCollection("HYCOM/sea_surface_elevation")
    .filterDate("2000-01-01", "2024-09-06")
    .select("surface_elevation")
)

print("Available image count:", collection.size().getInfo())

Available image count: 24382


In [6]:
# Convert catalog values to meters using scale factor 0.001
def convert_to_meters(image):
    return (
        image.multiply(0.001)
        .rename("elevation_m")
        .copyProperties(image, ["system:time_start"])
    )

elevation = collection.map(convert_to_meters)


In [7]:
# 4. Generate all months from 2000 to 2024
# ==========================================
start_date = ee.Date("2000-01-01")
end_date = ee.Date("2024-09-06")

total_months = end_date.difference(
    start_date, "month"
).ceil()

month_indices = ee.List.sequence(
    0, total_months.subtract(1)
)



In [8]:
# 5. Calculate monthly mean
# ==========================================
def calculate_monthly_mean(index):
    month_start = start_date.advance(index, "month")
    month_end = month_start.advance(1, "month")

    monthly_collection = elevation.filterDate(
        month_start, month_end
    )

    image_count = monthly_collection.size()

    monthly_image = monthly_collection.mean()

    stats = monthly_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=9000,
        maxPixels=1e8,
        bestEffort=True
    )

    return ee.Feature(None, {
        "Year": month_start.get("year"),
        "Month": month_start.get("month"),
        "Date": month_start.format("YYYY-MM"),
        "Mean_Elevation_Anomaly_m": stats.get("elevation_m"),
        "Available_Image_Count": image_count
    })

monthly_data = ee.FeatureCollection(
    month_indices.map(calculate_monthly_mean)
)

# Remove months without observations
monthly_data = monthly_data.filter(
    ee.Filter.gt("Available_Image_Count", 0)
).filter(
    ee.Filter.notNull(["Mean_Elevation_Anomaly_m"])
)


In [9]:
# 6. Retrieve results in Python
# ==========================================
result = monthly_data.getInfo()

rows = [
    feature["properties"]
    for feature in result["features"]
]

df = pd.DataFrame(rows)

df = df.sort_values(
    ["Year", "Month"]
).reset_index(drop=True)


In [10]:
# 7. Export to Excel
# ==========================================
output_file = "HYCOM_Sundarbans_Monthly_2000_2024.xlsx"

with pd.ExcelWriter(
    output_file, engine="openpyxl"
) as writer:
    df.to_excel(
        writer,
        sheet_name="Monthly_Data",
        index=False
    )

print("\nMonthly Excel file created successfully!")
print("File:", output_file)
print("Number of monthly records:", len(df))
print(df.head(12).to_string(index=False))



Monthly Excel file created successfully!
File: HYCOM_Sundarbans_Monthly_2000_2024.xlsx
Number of monthly records: 297
 Available_Image_Count    Date  Mean_Elevation_Anomaly_m  Month  Year
                    31 2000-01                  0.407380      1  2000
                    29 2000-02                  0.376481      2  2000
                    31 2000-03                  0.445538      3  2000
                    30 2000-04                  0.532423      4  2000
                    28 2000-05                  0.621180      5  2000
                    30 2000-06                  0.645706      6  2000
                    31 2000-07                  0.648830      7  2000
                    31 2000-08                  0.642987      8  2000
                    30 2000-09                  0.573857      9  2000
                    31 2000-10                  0.613979     10  2000
                    30 2000-11                  0.603771     11  2000
                    31 2000-12           

In [13]:
## Sea Surface Elevation Anomaly EDA

In [19]:
input_file = "HYCOM_Sundarbans_Monthly_2000_2024.xlsx"

In [20]:
os.makedirs("EDA", exist_ok=True)

In [21]:
output_file = os.path.join(
    "EDA",
    "Sea_Level_Anomaly_EDA_2_1_Descriptive_Statistics.xlsx"
)

In [17]:
## LOAD DATA

In [25]:
df = pd.read_excel(
    input_file,
    sheet_name="Monthly_Data"
)


In [26]:
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

In [27]:
df["Mean_Elevation_Anomaly_m"] = pd.to_numeric(
    df["Mean_Elevation_Anomaly_m"],
    errors="coerce"
)

In [28]:
df["Available_Image_Count"] = pd.to_numeric(
    df["Available_Image_Count"],
    errors="coerce"
)

In [29]:
elevation = df["Mean_Elevation_Anomaly_m"].dropna()

In [30]:
## DATASET OVERVIEW

In [31]:
overview = pd.DataFrame({
    "Metric": [
        "Total Observations",
        "Total Variables",
        "Start Date",
        "End Date",
        "Missing Elevation Anomaly Values",
        "Duplicate Records",
        "Duplicate Dates",
        "Minimum Elevation Anomaly (m)",
        "Maximum Elevation Anomaly (m)",
        "Mean Elevation Anomaly (m)",
        "Median Elevation Anomaly (m)"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["Date"].min(),
        df["Date"].max(),
        df["Mean_Elevation_Anomaly_m"].isna().sum(),
        df.duplicated().sum(),
        df["Date"].duplicated().sum(),
        elevation.min(),
        elevation.max(),
        elevation.mean(),
        elevation.median()
    ]
})


In [32]:
## DESCRIPTIVE STATISTICS

In [33]:
descriptive_stats = elevation.describe().to_frame(
    name="Elevation_Anomaly_m"
)


In [34]:
descriptive_stats.loc["Range"] = (
    elevation.max() - elevation.min()
)

In [35]:
descriptive_stats.loc["IQR"] = (
    elevation.quantile(0.75) -
    elevation.quantile(0.25)
)

In [36]:
descriptive_stats.loc["Variance"] = elevation.var()

descriptive_stats.loc["Standard Deviation"] = elevation.std()

In [37]:
## DISTRIBUTION SHAPE

In [38]:
distribution = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Skewness",
        "Kurtosis"
    ],
    "Value": [
        elevation.mean(),
        elevation.median(),
        elevation.skew(),
        elevation.kurt()
    ]
})

In [39]:
## IQR-BASED POTENTIAL OUTLIER DETECTION

In [40]:
Q1 = elevation.quantile(0.25)
Q3 = elevation.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

In [41]:
df["Outlier_Status"] = np.select(
    [
        df["Mean_Elevation_Anomaly_m"].isna(),
        (df["Mean_Elevation_Anomaly_m"] < lower_bound) |
        (df["Mean_Elevation_Anomaly_m"] > upper_bound)
    ],
    [
        "Missing Value",
        "Potential Outlier"
    ],
    default="Normal"
)

In [42]:
outliers = df[
    df["Outlier_Status"] == "Potential Outlier"
].copy()

outlier_summary = pd.DataFrame({
    "Metric": [
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound",
        "Total Potential Outliers"
    ],
    "Value": [
        Q1,
        Q3,
        IQR,
        lower_bound,
        upper_bound,
        len(outliers)
    ]
})

In [43]:
## MONTHLY SUMMARY

In [44]:
monthly_summary = (
    df.groupby("Month")["Mean_Elevation_Anomaly_m"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
    .sort_values("Month")
)

In [45]:
## ANNUAL SUMMARY

In [46]:
annual_summary = (
    df.groupby("Year")["Mean_Elevation_Anomaly_m"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Standard_Deviation="std",
        Minimum="min",
        Maximum="max"
    )
    .reset_index()
)

In [47]:
## IMAGE AVAILABILITY SUMMARY

In [48]:
image_summary = pd.DataFrame({
    "Metric": [
        "Total Available Images",
        "Average Images Per Month",
        "Minimum Images Per Month",
        "Maximum Images Per Month",
        "Months With Zero Images",
        "Months With One Or More Images"
    ],
    "Value": [
        df["Available_Image_Count"].sum(),
        df["Available_Image_Count"].mean(),
        df["Available_Image_Count"].min(),
        df["Available_Image_Count"].max(),
        (df["Available_Image_Count"] == 0).sum(),
        (df["Available_Image_Count"] > 0).sum()
    ]
})


In [49]:
## DATE COVERAGE CHECK

In [50]:
expected_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="MS"
)

actual_dates = pd.DatetimeIndex(
    df["Date"].dropna().unique()
).sort_values()

missing_dates = expected_dates.difference(actual_dates)

date_coverage = pd.DataFrame({
    "Metric": [
        "Expected Monthly Records",
        "Actual Unique Monthly Dates",
        "Missing Monthly Dates"
    ],
    "Value": [
        len(expected_dates),
        len(actual_dates),
        len(missing_dates)
    ]
})

missing_date_details = pd.DataFrame({
    "Missing_Date": missing_dates
})

In [51]:
## EXPORT RESULTS TO EXCEL

In [52]:
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    overview.to_excel(
        writer,
        sheet_name="Dataset_Overview",
        index=False
    )

    descriptive_stats.to_excel(
        writer,
        sheet_name="Descriptive_Statistics"
    )

    distribution.to_excel(
        writer,
        sheet_name="Distribution_Shape",
        index=False
    )

    outlier_summary.to_excel(
        writer,
        sheet_name="Outlier_Summary",
        index=False
    )

    outliers.to_excel(
        writer,
        sheet_name="Potential_Outliers",
        index=False
    )

    monthly_summary.to_excel(
        writer,
        sheet_name="Monthly_Summary",
        index=False
    )

    annual_summary.to_excel(
        writer,
        sheet_name="Annual_Summary",
        index=False
    )

    image_summary.to_excel(
        writer,
        sheet_name="Image_Availability",
        index=False
    )

    date_coverage.to_excel(
        writer,
        sheet_name="Date_Coverage",
        index=False
    )

    missing_date_details.to_excel(
        writer,
        sheet_name="Missing_Dates",
        index=False
    )

In [53]:
print("Sea Surface Elevation Anomaly EDA 2.1 completed!")
print("Total observations:", len(df))
print("Valid observations:", elevation.count())
print("Missing values:", df["Mean_Elevation_Anomaly_m"].isna().sum())
print("Potential outliers:", len(outliers))
print("Missing monthly dates:", len(missing_dates))
print("Report saved at:", os.path.abspath(output_file))

Sea Surface Elevation Anomaly EDA 2.1 completed!
-----------------------------------------------
Total observations: 297
Valid observations: 297
Missing values: 0
Potential outliers: 0
Missing monthly dates: 0
-----------------------------------------------
Report saved at: C:\projects_data\sundarbans_climate_analysis_gee\EDA\Sea_Level_Anomaly_EDA_2_1_Descriptive_Statistics.xlsx
